# Step 04 — Deep Learning pour la classification de santé mentale

Ce notebook constitue la suite logique de :

- **Step 01 : Data Understanding / EDA**
- **Step 02 : Data Cleaning & Preprocessing**
- **Step 03 : Feature Engineering + Machine Learning classique**

Dans le Step 03, les approches classiques ont été testées sérieusement : TF-IDF, Word TF-IDF, Character TF-IDF, Word+Char TF-IDF, Word2Vec, Logistic Regression, Linear SVM, Random Forest, XGBoost, class weights, oversampling, undersampling, SMOTE et GridSearch.

Les résultats se sont stabilisés autour de **0.68–0.70 en Macro F1**, ce qui indique que les méthodes classiques atteignent une limite sur ce dataset multiclasses. L’objectif du Step 04 est donc de tester une approche Deep Learning plus adaptée au texte séquentiel : **BiLSTM**.

## 1. Objectif de cette étape

Contrairement aux modèles classiques, qui représentent le texte comme un sac de mots, les modèles séquentiels comme LSTM/BiLSTM prennent en compte l’ordre des mots.

Exemple :

- `I want to die`
- `I do not want to die`

Avec TF-IDF, ces deux phrases partagent beaucoup de mots. Avec un modèle séquentiel, l’ordre et la négation peuvent être mieux capturés.

Dans ce notebook, on va tester :

1. Un modèle Deep Learning simple : **Embedding + GlobalAveragePooling**
2. Un modèle principal : **BiLSTM**
3. Une version améliorée : **CNN + BiLSTM**
4. Une comparaison avec les résultats classiques du Step 03

## 2. Imports et configuration

Cette cellule charge les bibliothèques nécessaires. Si TensorFlow n’est pas installé, il faudra l’installer avec :

```python
!pip install tensorflow
```

In [ ]:
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Dropout, Bidirectional, LSTM, GlobalAveragePooling1D, Conv1D, MaxPooling1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)

### Interprétation

On fixe un `SEED` pour rendre les résultats plus reproductibles. En Deep Learning, les résultats peuvent varier légèrement d’une exécution à l’autre à cause de l’initialisation aléatoire des poids et de l’optimisation.

## 3. Chargement du dataset nettoyé

On utilise directement le fichier produit par Step 02 : `mental_health_cleaned.csv`.

Important : on utilise la colonne **`clean_text`**, pas `statement`, car le nettoyage a déjà été fait dans Step 02.

In [ ]:
DATA_PATH = "mental_health_cleaned.csv"
TEXT_COL = "clean_text"
LABEL_COL = "status"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Le fichier {DATA_PATH} est introuvable. Place-le dans le même dossier que ce notebook."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset chargé avec succès.")
print("Dimensions initiales :", df.shape)
print("Colonnes disponibles :", df.columns.tolist())

display(df.head())

## 4. Vérification finale avant Deep Learning

Avant d’entraîner un réseau de neurones, il faut vérifier :

- absence de textes vides,
- absence de labels manquants,
- longueur minimale des textes,
- distribution des classes.

Cette étape évite les erreurs pendant la tokenisation et l’entraînement.

In [ ]:
df_dl = df[[TEXT_COL, LABEL_COL]].copy()

df_dl[TEXT_COL] = df_dl[TEXT_COL].fillna("").astype(str).str.strip()
df_dl[LABEL_COL] = df_dl[LABEL_COL].fillna("").astype(str).str.strip()

df_dl["num_words"] = df_dl[TEXT_COL].apply(lambda x: len(x.split()))

# On garde les textes ayant au moins 3 mots pour éviter les séquences presque vides.
df_dl = df_dl[df_dl["num_words"] >= 3].copy()

print("Dimensions après vérification :", df_dl.shape)
print("Textes vides restants :", (df_dl[TEXT_COL] == "").sum())
print("Nombre de classes :", df_dl[LABEL_COL].nunique())

display(df_dl[LABEL_COL].value_counts())

### Interprétation

Le dataset utilisé pour le Deep Learning doit rester cohérent avec celui utilisé dans Step 03. On ne refait pas un nouveau nettoyage ici : on vérifie seulement que les données sont exploitables pour l’entraînement.

## 5. Distribution des classes

Même si on utilise un modèle Deep Learning, le déséquilibre des classes reste important. On va donc utiliser `class_weight` pendant l’entraînement.

In [ ]:
class_counts = df_dl[LABEL_COL].value_counts()
class_percent = df_dl[LABEL_COL].value_counts(normalize=True) * 100

class_summary = pd.DataFrame({
    "classe": class_counts.index,
    "nombre_exemples": class_counts.values,
    "pourcentage": class_percent.values.round(2)
})

display(class_summary)

plt.figure(figsize=(10, 5))
sns.countplot(data=df_dl, x=LABEL_COL, order=class_counts.index)
plt.title("Distribution des classes - Dataset nettoyé")
plt.xlabel("Classe")
plt.ylabel("Nombre d'exemples")
plt.xticks(rotation=45)
plt.show()

## 6. Split train / validation / test

On utilise un split stratifié pour conserver la même distribution de classes dans train, validation et test.

Important : la validation et le test ne doivent jamais être équilibrés artificiellement. Ils doivent représenter la vraie difficulté du dataset.

In [ ]:
X = df_dl[TEXT_COL].values
y = df_dl[LABEL_COL].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("Train :", X_train.shape)
print("Validation :", X_val.shape)
print("Test :", X_test.shape)

## 7. Encodage des labels

Les réseaux de neurones ne travaillent pas directement avec des labels textuels. On transforme donc les classes en entiers, puis en format `one-hot`.

In [ ]:
label_encoder = LabelEncoder()

y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)

num_classes = len(label_encoder.classes_)

print("Classes :")
for i, cls in enumerate(label_encoder.classes_):
    print(i, "->", cls)

# One-hot encoding pour categorical_crossentropy
y_train_cat = tf.keras.utils.to_categorical(y_train_enc, num_classes=num_classes)
y_val_cat = tf.keras.utils.to_categorical(y_val_enc, num_classes=num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test_enc, num_classes=num_classes)

## 8. Gestion du déséquilibre avec class weights

Au lieu de rééchantillonner les textes, on donne plus de poids aux classes minoritaires pendant l’apprentissage.

Cela permet au modèle de ne pas optimiser uniquement les classes majoritaires.

In [ ]:
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_enc),
    y=y_train_enc
)

class_weights = {
    i: weight
    for i, weight in enumerate(class_weights_array)
}

print("Class weights :")
for idx, weight in class_weights.items():
    print(label_encoder.classes_[idx], "->", round(weight, 3))

## 9. Tokenisation du texte

Contrairement à TF-IDF, un modèle Deep Learning transforme chaque texte en une séquence d’indices numériques.

Exemple :

`i feel very sad` → `[14, 52, 88, 231]`

Chaque mot correspond à un identifiant dans un vocabulaire appris sur le train uniquement.

In [ ]:
VOCAB_SIZE = 30000
OOV_TOKEN = "<OOV>"

# Le tokenizer est appris uniquement sur X_train pour éviter la fuite de données.
tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token=OOV_TOKEN
)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

word_index_size = len(tokenizer.word_index)

print("Taille réelle du vocabulaire :", word_index_size)
print("Taille maximale utilisée :", VOCAB_SIZE)
print("Exemple de séquence :")
print(X_train_seq[0][:30])

## 10. Choix de la longueur maximale des séquences

Les textes n’ont pas tous la même longueur. Pour entraîner un réseau de neurones, toutes les séquences doivent avoir la même taille.

On choisit une longueur maximale basée sur les percentiles afin d’éviter que quelques textes très longs imposent une séquence énorme.

In [ ]:
seq_lengths = [len(seq) for seq in X_train_seq]

length_summary = pd.Series(seq_lengths).describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])
display(length_summary)

MAX_LEN = int(np.percentile(seq_lengths, 95))
MAX_LEN = max(MAX_LEN, 30)

print("Longueur maximale choisie MAX_LEN =", MAX_LEN)

plt.figure(figsize=(10, 5))
sns.histplot(seq_lengths, bins=60)
plt.axvline(MAX_LEN, color="red", linestyle="--", label=f"MAX_LEN = {MAX_LEN}")
plt.title("Distribution des longueurs de séquences")
plt.xlabel("Nombre de tokens")
plt.ylabel("Nombre de textes")
plt.legend()
plt.show()

### Interprétation

On ne choisit pas forcément la longueur maximale absolue, car certains textes sont très longs et peuvent ralentir fortement l’entraînement. Le 95e percentile est un compromis : il garde la majorité des textes tout en limitant le coût de calcul.

## 11. Padding des séquences

Le padding ajoute des zéros à la fin des séquences courtes et coupe les séquences trop longues.

In [ ]:
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

print("X_train_pad shape :", X_train_pad.shape)
print("X_val_pad shape :", X_val_pad.shape)
print("X_test_pad shape :", X_test_pad.shape)

## 12. Fonction d’évaluation commune

Cette fonction permet d’évaluer tous les modèles Deep Learning avec les mêmes métriques : Accuracy, Precision Macro, Recall Macro, Macro F1 et Weighted F1.

In [ ]:
def evaluate_dl_model(model_name, model, X_eval, y_true_text, y_true_enc):
    y_proba = model.predict(X_eval, verbose=0)
    y_pred_enc = np.argmax(y_proba, axis=1)
    y_pred_text = label_encoder.inverse_transform(y_pred_enc)

    result = {
        "model": model_name,
        "accuracy": accuracy_score(y_true_text, y_pred_text),
        "precision_macro": precision_score(y_true_text, y_pred_text, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true_text, y_pred_text, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true_text, y_pred_text, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true_text, y_pred_text, average="weighted", zero_division=0)
    }

    print("Classification report -", model_name)
    print(classification_report(y_true_text, y_pred_text, zero_division=0))

    return result, y_pred_text, y_proba

## 13. Callbacks d’entraînement

On utilise :

- `EarlyStopping` : arrête l’entraînement si le modèle n’améliore plus la validation.
- `ReduceLROnPlateau` : réduit le learning rate si la progression bloque.
- `ModelCheckpoint` : sauvegarde le meilleur modèle.

Cela permet d’éviter le surapprentissage et d’améliorer la stabilité.

In [ ]:
os.makedirs("models", exist_ok=True)

callbacks_common = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
]

## 14. Modèle 1 — Baseline Deep Learning simple

Ce modèle utilise :

- une couche `Embedding`,
- une moyenne globale des embeddings,
- une couche Dense,
- une sortie Softmax.

Il sert de baseline Deep Learning. Il est plus simple qu’un BiLSTM, mais permet déjà de comparer avec les modèles classiques.

In [ ]:
EMBEDDING_DIM = 128

simple_dl_model = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_LEN
    ),
    GlobalAveragePooling1D(),
    Dropout(0.4),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(num_classes, activation="softmax")
])

simple_dl_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

simple_dl_model.summary()

In [ ]:
print("Entraînement du modèle Deep Learning simple...")

history_simple = simple_dl_model.fit(
    X_train_pad,
    y_train_cat,
    validation_data=(X_val_pad, y_val_cat),
    epochs=15,
    batch_size=64,
    class_weight=class_weights,
    callbacks=callbacks_common,
    verbose=1
)

## 15. Courbes d’apprentissage — modèle simple

Les courbes permettent de voir si le modèle apprend correctement ou s’il surapprend.

In [ ]:
def plot_history(history, title):
    hist = pd.DataFrame(history.history)

    plt.figure(figsize=(10, 5))
    plt.plot(hist["loss"], label="train_loss")
    plt.plot(hist["val_loss"], label="val_loss")
    plt.title(title + " - Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    if "accuracy" in hist.columns:
        plt.figure(figsize=(10, 5))
        plt.plot(hist["accuracy"], label="train_accuracy")
        plt.plot(hist["val_accuracy"], label="val_accuracy")
        plt.title(title + " - Accuracy")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.legend()
        plt.show()

plot_history(history_simple, "Modèle simple")

In [ ]:
simple_result, y_pred_simple, y_proba_simple = evaluate_dl_model(
    "Embedding + GlobalAveragePooling",
    simple_dl_model,
    X_val_pad,
    y_val,
    y_val_enc
)

simple_result

## 16. Modèle 2 — BiLSTM

Le BiLSTM lit le texte dans les deux sens :

- de gauche à droite,
- de droite à gauche.

Cela permet de mieux capturer le contexte et les relations entre les mots, ce qui est important dans les textes de santé mentale.

In [ ]:
bilstm_model = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_LEN
    ),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.5),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(num_classes, activation="softmax")
])

bilstm_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

bilstm_model.summary()

In [ ]:
print("Entraînement du modèle BiLSTM...")

history_bilstm = bilstm_model.fit(
    X_train_pad,
    y_train_cat,
    validation_data=(X_val_pad, y_val_cat),
    epochs=15,
    batch_size=64,
    class_weight=class_weights,
    callbacks=callbacks_common,
    verbose=1
)

In [ ]:
plot_history(history_bilstm, "BiLSTM")

bilstm_result, y_pred_bilstm, y_proba_bilstm = evaluate_dl_model(
    "BiLSTM",
    bilstm_model,
    X_val_pad,
    y_val,
    y_val_enc
)

bilstm_result

## 17. Modèle 3 — CNN + BiLSTM

Cette architecture combine :

- `Conv1D` pour détecter des motifs locaux dans le texte,
- `BiLSTM` pour apprendre la séquence globale.

Elle peut être utile pour les textes de réseaux sociaux, où certaines expressions courtes sont très informatives.

In [ ]:
cnn_bilstm_model = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_LEN
    ),
    Conv1D(filters=128, kernel_size=5, activation="relu"),
    MaxPooling1D(pool_size=2),
    Bidirectional(LSTM(64)),
    Dropout(0.5),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(num_classes, activation="softmax")
])

cnn_bilstm_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_bilstm_model.summary()

In [ ]:
print("Entraînement du modèle CNN + BiLSTM...")

history_cnn_bilstm = cnn_bilstm_model.fit(
    X_train_pad,
    y_train_cat,
    validation_data=(X_val_pad, y_val_cat),
    epochs=15,
    batch_size=64,
    class_weight=class_weights,
    callbacks=callbacks_common,
    verbose=1
)

In [ ]:
plot_history(history_cnn_bilstm, "CNN + BiLSTM")

cnn_bilstm_result, y_pred_cnn_bilstm, y_proba_cnn_bilstm = evaluate_dl_model(
    "CNN + BiLSTM",
    cnn_bilstm_model,
    X_val_pad,
    y_val,
    y_val_enc
)

cnn_bilstm_result

## 18. Comparaison des modèles Deep Learning

On compare les modèles selon les mêmes métriques que dans Step 03.

La métrique principale reste **Macro F1**, car le dataset est multiclasses et déséquilibré.

In [ ]:
dl_results = pd.DataFrame([
    simple_result,
    bilstm_result,
    cnn_bilstm_result
])

metrics_cols = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted"]
for col in metrics_cols:
    dl_results[col] = dl_results[col].round(4)

dl_results = dl_results.sort_values(by="f1_macro", ascending=False)

display(dl_results)

plt.figure(figsize=(9, 5))
sns.barplot(data=dl_results, x="model", y="f1_macro")
plt.title("Comparaison des modèles Deep Learning — Macro F1")
plt.xlabel("Modèle")
plt.ylabel("Macro F1")
plt.xticks(rotation=30)
plt.ylim(0, 1)
plt.show()

## 19. Sélection du meilleur modèle Deep Learning

On sélectionne le modèle ayant le meilleur Macro F1 sur la validation.

In [ ]:
best_dl_name = dl_results.iloc[0]["model"]
print("Meilleur modèle Deep Learning sur validation :", best_dl_name)

if best_dl_name == "Embedding + GlobalAveragePooling":
    best_dl_model = simple_dl_model
    best_dl_pred_val = y_pred_simple
elif best_dl_name == "BiLSTM":
    best_dl_model = bilstm_model
    best_dl_pred_val = y_pred_bilstm
else:
    best_dl_model = cnn_bilstm_model
    best_dl_pred_val = y_pred_cnn_bilstm

## 20. Matrice de confusion du meilleur modèle

Cette matrice permet d’identifier les classes encore confondues après passage au Deep Learning.

In [ ]:
cm = confusion_matrix(y_val, best_dl_pred_val, labels=label_encoder.classes_)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.title("Matrice de confusion — Meilleur modèle Deep Learning")
plt.xlabel("Classe prédite")
plt.ylabel("Classe réelle")
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.show()

## 21. Analyse des classes faibles

On identifie les classes dont le F1-score reste le plus faible. Cela permet d’expliquer les limites du modèle aux encadrants.

In [ ]:
report_best = classification_report(
    y_val,
    best_dl_pred_val,
    output_dict=True,
    zero_division=0
)

report_best_df = pd.DataFrame(report_best).T
class_report = report_best_df.drop(index=["accuracy", "macro avg", "weighted avg"], errors="ignore")
weak_classes = class_report.sort_values(by="f1-score").head(3)

display(report_best_df.round(4))
print("Classes les plus faibles :")
display(weak_classes.round(4))

### Interprétation attendue

Si certaines classes restent faibles malgré le Deep Learning, cela signifie que la difficulté vient probablement de la séparabilité des labels ou d’un manque d’exemples réellement distinctifs pour ces classes.

Ce n’est pas forcément une erreur du modèle : cela peut être une limite du dataset.

## 22. Évaluation finale sur le test set

Le test set ne doit être utilisé qu’une seule fois, après avoir choisi le meilleur modèle sur validation.

In [ ]:
test_result, y_pred_test, y_proba_test = evaluate_dl_model(
    best_dl_name + " - Test",
    best_dl_model,
    X_test_pad,
    y_test,
    y_test_enc
)

test_result

## 23. Comparaison avec la meilleure baseline classique

Dans Step 03, le meilleur modèle classique était généralement autour de :

- Accuracy ≈ 0.73–0.74
- Macro F1 ≈ 0.68–0.70

Cette cellule permet de comparer directement les résultats obtenus avec le Deep Learning.

In [ ]:
classic_baseline = {
    "model": "Best classical baseline (TF-IDF + Linear SVM)",
    "accuracy": 0.74,
    "precision_macro": 0.70,
    "recall_macro": 0.68,
    "f1_macro": 0.69,
    "f1_weighted": 0.73
}

comparison_df = pd.DataFrame([
    classic_baseline,
    test_result
])

for col in metrics_cols:
    comparison_df[col] = comparison_df[col].round(4)

display(comparison_df)

plt.figure(figsize=(9, 5))
sns.barplot(data=comparison_df, x="model", y="f1_macro")
plt.title("Comparaison modèle classique vs Deep Learning")
plt.xlabel("Modèle")
plt.ylabel("Macro F1")
plt.xticks(rotation=25)
plt.ylim(0, 1)
plt.show()

## 24. Sauvegarde du meilleur modèle Deep Learning

On sauvegarde :

- le modèle entraîné,
- le tokenizer,
- le label encoder,
- les paramètres importants.

In [ ]:
import pickle

os.makedirs("models", exist_ok=True)

best_dl_model.save("models/best_deep_learning_model.keras")

with open("models/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("models/label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

config = {
    "TEXT_COL": TEXT_COL,
    "LABEL_COL": LABEL_COL,
    "VOCAB_SIZE": VOCAB_SIZE,
    "MAX_LEN": MAX_LEN,
    "EMBEDDING_DIM": EMBEDDING_DIM,
    "best_model": best_dl_name,
    "classes": label_encoder.classes_.tolist()
}

with open("models/deep_learning_config.pkl", "wb") as f:
    pickle.dump(config, f)

print("Modèle Deep Learning sauvegardé dans le dossier models/.")

## 25. Conclusion de Step 04

Cette étape permet de comparer les modèles classiques avec des modèles Deep Learning capables de mieux prendre en compte l’ordre des mots.

Si le Deep Learning améliore nettement le Macro F1, on pourra conclure que la représentation séquentielle du texte est plus adaptée que TF-IDF pour ce problème.

Si l’amélioration reste limitée, cela indique que le problème vient probablement davantage de la structure du dataset : labels proches, classes difficiles, bruit d’annotation ou ambiguïté sémantique.

Dans les deux cas, cette étape fournit une conclusion solide et défendable devant l’encadrant.

## 26. Pistes futures

Après cette étape, les pistes possibles sont :

1. Tester des embeddings pré-entraînés comme GloVe.
2. Tester une architecture plus avancée : Attention + BiLSTM.
3. Tester un modèle Transformer léger comme DistilBERT.
4. Réfléchir à une fusion de classes faibles si certaines classes restent très difficiles.
5. Construire une interface de prédiction pour la démonstration finale.